# 04. Real / Imaginary Split MP-PCA — Open QSM Dataset

`02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb`에서 생성한
**5-echo noisy complex GRE**를 Real과 Imaginary 성분으로 분리한 뒤,
각 성분에 MP-PCA를 **독립적으로** 적용한다.

현재 실험 원칙:

- 입력: `(X, Y, Z, 5)` noisy complex GRE
- Noise levels: 10%, 30%, 50%
- Real / Imaginary 각각 `(X, Y, Z, 5)`
- Real MP-PCA와 Imaginary MP-PCA를 별도로 수행
- **Brain mask는 denoising에 사용하지 않음**
- **전체 volume을 denoising**
- Brain mask는 향후 성능평가 ROI용으로만 보관
- Patch radius = 3 → `7 × 7 × 7`
- Denoising 후 `Real + j·Imag`로 complex GRE 재결합

Real/Imaginary에는 음수 값이 존재할 수 있으므로,
기존 연구 코드에서 사용한 **negative clipping 제거 버전 `localpca_dn.py`의 `mppca()`**를 사용한다.


In [ ]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

# Real / Imaginary에는 음수가 존재할 수 있으므로
# negative clipping을 제거한 custom MP-PCA 사용
from localpca_dn import mppca


### 환경설정

새로운 Open QSM noisy complex 데이터는 다음 폴더에서 읽는다.

`./data/noisy_data/`

입력 파일:

- `qsm_siemens_monopolar_noisy_10.npz`
- `qsm_siemens_monopolar_noisy_30.npz`
- `qsm_siemens_monopolar_noisy_50.npz`

서버 실행 시 이 notebook과 함께 `localpca_dn.py`가 같은 작업 경로에서 import 가능해야 한다.


In [ ]:
NOISE_LEVELS = [10, 30, 50]

# 졸업논문 공통 설정
PATCH_R = 3  # 7 x 7 x 7 spatial patch

NOISY_DIR = Path("./data/noisy_data")
OUT_DIR = Path("./data/denoised/real_imag")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Noisy directory :", NOISY_DIR.resolve())
print("Output directory:", OUT_DIR.resolve())
print("Patch radius    :", PATCH_R)
print(
    "Patch size      :",
    f"{2 * PATCH_R + 1} x {2 * PATCH_R + 1} x {2 * PATCH_R + 1}"
)


### 1. 새 QSM Noise 데이터 불러오기

각 noise level의 `.npz` 파일에서 `noisy_complex`를 불러온 뒤
Real과 Imaginary 성분을 분리한다.

입력 구조:

- `noisy_complex`: `(X, Y, Z, 5)` complex64
- `mask_brain`: `(X, Y, Z)` boolean mask — **평가용**
- `noise_ratio`
- `sigma`
- `seed`
- `overall_mean`
- `echo_times`
- `field_strength`

`mask_brain`은 이 notebook에서 MP-PCA에 전달하지 않는다.


In [ ]:
noisy_datasets = {}

for noise_level in NOISE_LEVELS:
    noisy_path = (
        NOISY_DIR /
        f"qsm_siemens_monopolar_noisy_{noise_level}.npz"
    )

    if not noisy_path.exists():
        raise FileNotFoundError(
            f"Noise 파일을 찾을 수 없습니다: {noisy_path}\n"
            "먼저 02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb를 실행하세요."
        )

    data = np.load(noisy_path, allow_pickle=False)

    noisy_complex = data["noisy_complex"].astype(np.complex64)
    mask = data["mask_brain"].astype(bool)

    if noisy_complex.ndim != 4:
        raise ValueError(
            f"{noise_level}% noisy_complex는 4D여야 합니다. "
            f"현재 shape={noisy_complex.shape}"
        )

    if noisy_complex.shape[-1] != 5:
        raise ValueError(
            f"{noise_level}% 데이터는 5 echoes를 예상합니다. "
            f"실제={noisy_complex.shape[-1]}"
        )

    if mask.shape != noisy_complex.shape[:3]:
        raise ValueError(
            f"{noise_level}% mask shape {mask.shape} != "
            f"spatial shape {noisy_complex.shape[:3]}"
        )

    noisy_real = noisy_complex.real.astype(np.float32)
    noisy_imag = noisy_complex.imag.astype(np.float32)

    noisy_datasets[noise_level] = {
        "noisy_complex": noisy_complex,
        "noisy_real": noisy_real,
        "noisy_imag": noisy_imag,
        "mask_brain": mask,
        "noise_ratio": float(data["noise_ratio"]),
        "sigma": float(data["sigma"]),
        "seed": int(data["seed"]),
        "overall_mean": float(data["overall_mean"]),
        "echo_times": (
            data["echo_times"].astype(np.float64)
            if "echo_times" in data.files
            else np.full(5, np.nan, dtype=np.float64)
        ),
        "field_strength": (
            float(data["field_strength"])
            if "field_strength" in data.files
            else np.nan
        ),
    }

    print(
        f"{noise_level}% 로드 완료 | "
        f"complex={noisy_complex.shape}, "
        f"real={noisy_real.shape}, imag={noisy_imag.shape}"
    )


### 2. Real MP-PCA Denoising

각 noise level에서 Real 성분 `(X,Y,Z,5)`에 MP-PCA를 적용한다.

- Patch radius = 3
- Patch size = `7 × 7 × 7`
- Spatial voxels per patch = 343
- Features = 5 echoes
- 입력 구조 ≈ `343 × 5`
- **Mask 사용 안 함**
- **전체 volume 처리**


In [ ]:
deno_real_data = {}

for noise_level in NOISE_LEVELS:
    noisy_real = noisy_datasets[noise_level]["noisy_real"]

    print("\n" + "=" * 60)
    print(f"Noise level   : {noise_level}%")
    print("Component     : REAL")
    print(f"Input shape   : {noisy_real.shape}")
    print(f"Patch radius  : {PATCH_R}")
    print("Brain mask    : NOT USED")
    print("Denoise ROI   : FULL VOLUME")
    print("=" * 60)

    deno_real = mppca(
        noisy_real,
        patch_radius=PATCH_R
    ).astype(np.float32)

    if deno_real.shape != noisy_real.shape:
        raise ValueError(
            f"Real denoised shape {deno_real.shape} != input {noisy_real.shape}"
        )

    deno_real_data[noise_level] = deno_real

    print(f"{noise_level}% Real MP-PCA 완료")


### 3. Imaginary MP-PCA Denoising

Imaginary 성분도 Real과 동일한 조건으로 별도로 MP-PCA를 수행한다.

Real과 Imaginary는 서로 합쳐서 PCA를 수행하는 것이 아니라,
**각각 독립적인 5-echo MP-PCA**를 수행한다.


In [ ]:
deno_imag_data = {}

for noise_level in NOISE_LEVELS:
    noisy_imag = noisy_datasets[noise_level]["noisy_imag"]

    print("\n" + "=" * 60)
    print(f"Noise level   : {noise_level}%")
    print("Component     : IMAGINARY")
    print(f"Input shape   : {noisy_imag.shape}")
    print(f"Patch radius  : {PATCH_R}")
    print("Brain mask    : NOT USED")
    print("Denoise ROI   : FULL VOLUME")
    print("=" * 60)

    deno_imag = mppca(
        noisy_imag,
        patch_radius=PATCH_R
    ).astype(np.float32)

    if deno_imag.shape != noisy_imag.shape:
        raise ValueError(
            f"Imag denoised shape {deno_imag.shape} != input {noisy_imag.shape}"
        )

    deno_imag_data[noise_level] = deno_imag

    print(f"{noise_level}% Imaginary MP-PCA 완료")


### 4. Real / Imaginary 재결합

각각 denoising한 Real과 Imaginary를 다시 결합하여 complex GRE를 만든다.

`deno_complex = deno_real + j × deno_imag`

최종 shape은 다시 `(X, Y, Z, 5)` complex 데이터가 된다.


In [ ]:
denoised_data = {}

for noise_level in NOISE_LEVELS:
    deno_real = deno_real_data[noise_level]
    deno_imag = deno_imag_data[noise_level]

    deno_complex = (
        deno_real.astype(np.complex64)
        + 1j * deno_imag.astype(np.complex64)
    ).astype(np.complex64)

    denoised_data[noise_level] = {
        "deno_real": deno_real,
        "deno_imag": deno_imag,
        "deno_complex": deno_complex,
    }

    print(
        f"{noise_level}% complex 재결합 완료 | "
        f"shape={deno_complex.shape} | dtype={deno_complex.dtype}"
    )


### 5. 결과 저장

각 noise level의 Real/Imag Split MP-PCA 결과를 `.mat`으로 저장한다.

저장 항목:

- `deno_real`
- `deno_imag`
- `deno_complex`
- `mask_brain` — 향후 **평가 ROI용**
- `noise_level`
- `noise_ratio`
- `sigma`
- `seed`
- `overall_mean`
- `patch_radius`
- `echo_times`
- `field_strength`
- `denoise_region = "full_volume"`
- `mask_usage = "evaluation_only"`


In [ ]:
for noise_level in NOISE_LEVELS:
    meta = noisy_datasets[noise_level]

    deno_real = denoised_data[noise_level]["deno_real"]
    deno_imag = denoised_data[noise_level]["deno_imag"]
    deno_complex = denoised_data[noise_level]["deno_complex"]

    output_path = (
        OUT_DIR /
        f"denoised_real_imag_{noise_level}_r{PATCH_R}.mat"
    )

    save_data = {
        "deno_real": deno_real,
        "deno_imag": deno_imag,
        "deno_complex": deno_complex,
        "mask_brain": meta["mask_brain"].astype(np.uint8),
        "noise_level": np.int32(noise_level),
        "noise_ratio": np.float32(meta["noise_ratio"]),
        "sigma": np.float32(meta["sigma"]),
        "seed": np.int32(meta["seed"]),
        "overall_mean": np.float32(meta["overall_mean"]),
        "patch_radius": np.int32(PATCH_R),
        "echo_times": np.asarray(meta["echo_times"], dtype=np.float64),
        "field_strength": np.float32(meta["field_strength"]),
        "denoise_region": "full_volume",
        "mask_usage": "evaluation_only",
        "method": "real_imag_split_mppca",
    }

    sio.savemat(
        output_path,
        save_data,
        do_compression=True
    )

    print("저장 완료 ->", output_path)


### 6. 결과 확인

Clean reference를 여기서 직접 사용하지 않고 다음을 확인한다.

- Noisy Magnitude
- Denoised Magnitude
- Removed component = `Noisy - Denoised`

`Removed` 영상은 최종 error map이 아니라,
Real/Imag Split MP-PCA가 noisy 입력에서 제거한 성분을 확인하기 위한 시각화이다.

최종 PSNR / SSIM / NRMSE / HFEN 계산은 별도 평가 단계에서
clean reference와 `mask_brain`을 사용한다.


In [ ]:
echo_idx = 0
sample_shape = noisy_datasets[NOISE_LEVELS[0]]["noisy_complex"].shape
slice_idx = sample_shape[2] // 2

for noise_level in NOISE_LEVELS:
    noisy_complex = noisy_datasets[noise_level]["noisy_complex"]
    deno_complex = denoised_data[noise_level]["deno_complex"]

    noisy_mag = np.abs(noisy_complex).astype(np.float32)
    deno_mag = np.abs(deno_complex).astype(np.float32)

    noisy_2d = noisy_mag[:, :, slice_idx, echo_idx]
    deno_2d = deno_mag[:, :, slice_idx, echo_idx]
    removed_2d = noisy_2d - deno_2d

    vmin, vmax = np.percentile(noisy_2d, (1, 99))

    dv = float(np.percentile(np.abs(removed_2d), 99))
    dv = max(dv, 1e-8)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    ax[0].imshow(
        noisy_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[0].set_title(f"Noisy ({noise_level}%)")

    ax[1].imshow(
        deno_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[1].set_title("Real/Imag Split MP-PCA")

    ax[2].imshow(
        removed_2d,
        cmap="bwr",
        vmin=-dv,
        vmax=dv
    )
    ax[2].set_title("Removed (Noisy - Denoised)")

    for a in ax:
        a.axis("off")

    fig.suptitle(
        f"Full-volume Real/Imag Split MP-PCA | "
        f"Noise {noise_level}% | Echo {echo_idx + 1} | Slice {slice_idx}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()
